# Retail Inventory AI — Detection Checkpoint (Week 3)

## YOLOv8m on SKU-110K — full detection pipeline & Before/After comparison

_Project: Intelligent Retail Product Detection & Inventory Management_
_Date: 2026-06-07_

This notebook documents the **entire Module-1 detection pipeline** end to end, and proves the
checkpoint with a **before vs. after** comparison on the same shelf images.

It is organised in three parts:

| Part | Contents | Runs here? |
|------|----------|------------|
| **1. Data pipeline**   | How SKU-110K was downloaded and preprocessed (CSV → YOLO format) | 📄 **Documentation only** |
| **2. Training**        | The full GCP automation: launch the L4 VM → train → validate → sync → self-delete | 📄 **Documentation only** |
| **3. Before / After**  | COCO-pretrained vs. our fine-tuned `best.pt` on identical images + metrics | ▶️ **Runs** |

> **Why parts 1–2 do not run here:** the dataset is ~13 GB and training takes ~9.5 h on an
> L4 GPU — reproducing them in Colab would be slow, expensive, and pointless. Those steps
> already ran on GCP. Below we *show the exact code we used* so the pipeline is fully
> transparent and reproducible, then **execute only the lightweight inference** needed to
> demonstrate the result.

**Detection targets:** mAP@0.5 ≥ 0.70 · Precision ≥ 0.75 · Recall ≥ 0.70.
**Achieved:** mAP@0.5 = **0.9209** · Precision = **0.9193** · Recall = **0.8824** ✅

> **How to run:** Upload to Google Colab → Runtime → **Run all**. Only Part 3 executes
> (a few seconds of inference); Parts 1–2 are read-only documentation.

---
# Part 1 — Data Pipeline  📄 *(documentation, not executed)*

## 1.1  Dataset: SKU-110K

SKU-110K is a dense retail-shelf detection dataset from Goldman et al., CVPR 2019.

| Property             | Value                                              |
|----------------------|----------------------------------------------------|
| Images               | ~11,762 (train / val / test)                       |
| Annotations          | ~1.7 M bounding boxes                              |
| Avg objects / image  | ~147                                               |
| Classes              | 1 (`object` — localization only)                   |
| Source               | retail store shelves worldwide                     |
| License              | CC BY-NC 4.0 (research use)                        |

Why we chose it: it is the largest publicly-available **dense** retail dataset, with hundreds of
small objects per image — exactly the regime where stock COCO models fail and a fine-tuned
detector earns its keep.

## 1.2  How we downloaded it

We used the **Ultralytics built-in dataset spec**, which downloads *and* converts SKU-110K
automatically the first time it is referenced — no Kaggle account or manual step needed. Under
the hood it fetches the official `SKU110K_fixed.tar.gz` (~13 GB) and runs the CSV → YOLO
conversion described in 1.3.

**The one line that triggers the download** (inside the training command — see Part 2):

In [ ]:
# `data=SKU-110K.yaml` is an Ultralytics-bundled spec. On first use it:
#   1. downloads http://trax-geometry.s3.amazonaws.com/cvpr_challenge/SKU110K_fixed.tar.gz (~13 GB)
#   2. unpacks it to  /datasets/SKU-110K/{images,labels}/{train,val,test}
#   3. converts the CSV annotations to YOLO-format .txt label files (see 1.3)
#
# !yolo detect train model=yolov8m.pt data=SKU-110K.yaml ...


**Equivalent explicit/manual download** (what the spec does for you), shown for full
transparency in case the auto-spec is ever unavailable:

In [ ]:
# Manual alternative — fetch and unpack the raw dataset yourself.
# DOCUMENTATION ONLY — do not run here (~13 GB download).
#
# !mkdir -p /datasets && cd /datasets \
#   && wget http://trax-geometry.s3.amazonaws.com/cvpr_challenge/SKU110K_fixed.tar.gz \
#   && tar -xzf SKU110K_fixed.tar.gz
#
# Result tree:
#   /datasets/SKU110K_fixed/
#     ├── images/       # ~11,762 JPGs
#     └── annotations/
#         ├── annotations_train.csv
#         ├── annotations_val.csv
#         └── annotations_test.csv


## 1.3  Preprocessing — CSV → YOLO label format

SKU-110K ships annotations as **CSV**, one row per box:

```
image_name, x1, y1, x2, y2, class, image_width, image_height
test_0.jpg, 12,  34, 120, 210, object, 1920, 1080
test_0.jpg, 145, 30, 260, 215, object, 1920, 1080
...
```

YOLO needs **one `.txt` per image**, one row per box, with **normalised** `class cx cy w h`
(centre-x, centre-y, width, height — each in `[0, 1]`):

```
0  0.034375  0.112963  0.056250  0.162963
0  0.105729  0.113426  0.059896  0.171296
...
```

The conversion below mirrors what the Ultralytics `SKU-110K.yaml` spec runs automatically —
shown explicitly so a reviewer can see exactly what was applied to every image.

In [ ]:
# preprocess_sku110k.py  — CSV bounding boxes -> normalised YOLO .txt labels
# DOCUMENTATION ONLY — already executed by the Ultralytics spec on the GCP VM.

from pathlib import Path
import pandas as pd
from tqdm import tqdm

DATA = Path('/datasets/SKU110K_fixed')
COLS = ['image', 'x1', 'y1', 'x2', 'y2', 'class', 'img_w', 'img_h']

for split in ['train', 'val', 'test']:
    # Read the CSV — no header, columns in fixed order.
    df = pd.read_csv(DATA / 'annotations' / f'annotations_{split}.csv', names=COLS)

    out_dir = DATA / 'labels' / split
    out_dir.mkdir(parents=True, exist_ok=True)

    # All boxes belonging to the same image -> one .txt file.
    for image_name, g in tqdm(df.groupby('image'), desc=f'{split} labels'):
        lines = []
        for _, r in g.iterrows():
            # Convert pixel-space (x1,y1,x2,y2) to normalised YOLO (cx,cy,w,h).
            w  = (r.x2 - r.x1) / r.img_w           # width      (normalised)
            h  = (r.y2 - r.y1) / r.img_h           # height     (normalised)
            cx = (r.x1 + r.x2) / 2 / r.img_w       # centre-x   (normalised)
            cy = (r.y1 + r.y2) / 2 / r.img_h       # centre-y   (normalised)
            # Class id 0 = 'object' (single-class dataset).
            lines.append(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')

        # One label file per image, e.g. labels/train/train_0.txt
        (out_dir / f'{Path(image_name).stem}.txt').write_text('\n'.join(lines))


**Edge cases handled by Ultralytics during its label scan:**

- A handful of SKU-110K images have **corrupt/zero-area boxes** — Ultralytics skips them
  automatically. Our run dropped exactly **3 corrupt training images** (logged in `train.log`).
- A handful of JPEGs have **truncated EOI markers** ("Corrupt JPEG data: extraneous bytes…");
  these are auto-restored and saved back, so training continues unaffected.

## 1.4  The dataset spec — `SKU-110K.yaml`

The YOLO data spec ties images + labels together and declares the single class. Ultralytics
bundles this file; reproduced here for completeness:

In [ ]:
# SKU-110K.yaml  (Ultralytics-bundled — shown for transparency)
# DOCUMENTATION ONLY — Ultralytics writes this on first use.

yaml_spec = '''
path: ../datasets/SKU-110K       # dataset root
train: train.txt                 # list of training image paths
val:   val.txt                   # list of validation image paths
test:  test.txt                  # list of test image paths

names:
  0: object                      # single class — detection / localization only

# `download:` field holds the python that fetches the tar.gz and runs the
# CSV -> YOLO conversion from 1.3 the first time the spec is used.
'''
print(yaml_spec)


## 1.5  Image preprocessing at train/inference time

These transforms are applied **on-the-fly** by YOLO (not as a separate offline step):

| Step                    | Train split            | Val / Test split    |
|-------------------------|------------------------|---------------------|
| Letterbox resize        | 1280 × 1280            | 1280 × 1280         |
| Pixel normalisation     | divide by 255 → [0, 1] | same                |
| Mosaic augmentation     | ✓                      | ✗                   |
| HSV jitter              | ✓                      | ✗                   |
| Horizontal flip         | ✓                      | ✗                   |
| Random scale + translate| ✓                      | ✗                   |

**Why imgsz = 1280 and not the YOLO default 640?** SKU-110K averages ~147 tiny products per image.
At 640 px, individual SKUs collapse to a few pixels and recall craters. 1280 px keeps them
recognisable while still fitting on the L4's 24 GB VRAM (with AutoBatch landing on batch=3).

---
# Part 2 — Training Pipeline  📄 *(the exact GCP automation, not executed here)*

We fine-tuned **YOLOv8m** (COCO-pretrained) on SKU-110K on a single **NVIDIA L4** GPU
(`g2-standard-8`, 24 GB VRAM) for **~9.5 h**. The whole flow is fully automated:

```
  (laptop)                       (GCP)
  launch_vm.sh  ─────►  L4 VM boots
                            │
                            ▼
                 train_sku110k.sh runs as root startup
                            │
                            ├─ install libs + ultralytics
                            ├─ yolo detect train  (≤ 9.5 h cap)
                            ├─ yolo detect val
                            ├─ extract metrics.json
                            ├─ rsync /opt/runs -> GCS bucket
                            └─ delete self          ← cost safety
```

The two scripts — `detection/launch_vm.sh` and `detection/train_sku110k.sh` — are split into
the logical blocks below.

## 2.1  Launcher — create the L4 VM (`detection/launch_vm.sh`)

Runs **on the laptop** (not on GCP). It creates the VM, attaches an L4, picks the Deep-Learning
PyTorch image (CUDA + driver preinstalled), and uploads the training script as the boot
**startup-script** so training begins immediately on first boot — no SSH required.

### 2.1.1  Inputs and defaults

In [ ]:
# detection/launch_vm.sh  (top of file)
# DOCUMENTATION ONLY — already ran on the laptop.

# !/bin/bash
# set -euo pipefail
#
# # Required: pick the GCP project + bucket. Zone defaults to us-central1-a (L4 quota lives there).
# : "${PROJECT_ID:?set PROJECT_ID (e.g. export PROJECT_ID=my-gcp-project)}"
# ZONE="${ZONE:-us-central1-a}"
# BUCKET="${BUCKET:-gs://${PROJECT_ID}-sku110k-yolo}"
# INSTANCE="${INSTANCE:-sku110k-train}"


### 2.1.2  Inject the bucket name into the startup script

The training script lives in the repo with `__BUCKET__` as a placeholder. The launcher
substitutes the real bucket URI before uploading — keeps the bucket name out of source control.

In [ ]:
# Resolve this script's directory so it works whether we run it from
# the repo root or from detection/.
# SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
#
# # Replace __BUCKET__ in train_sku110k.sh with the real GCS URI.
# STARTUP="$(mktemp)"
# sed "s|__BUCKET__|${BUCKET}|g" "$SCRIPT_DIR/train_sku110k.sh" > "$STARTUP"


### 2.1.3  Create the VM

Key flags explained inline.

In [ ]:
# gcloud compute instances create "$INSTANCE" \
#   --project="$PROJECT_ID" \
#   --zone="$ZONE" \
#   --machine-type=g2-standard-8 \                 # 8 vCPU, 32 GB RAM (L4 host shape)
#   --accelerator=type=nvidia-l4,count=1 \         # 1 × NVIDIA L4 (24 GB VRAM)
#   --maintenance-policy=TERMINATE \               # GPUs require TERMINATE, not MIGRATE
#   --restart-on-failure \
#   --image-family=pytorch-2-9-cu129-ubuntu-2204-nvidia-580 \   # DLVM: torch+cuda+driver preinstalled
#   --image-project=deeplearning-platform-release \
#   --boot-disk-size=200GB \                       # ~13 GB dataset + checkpoints + room
#   --boot-disk-type=pd-ssd \                      # fast disk for image I/O during training
#   --metadata="install-nvidia-driver=True" \      # idempotent — DLVM already has it
#   --metadata-from-file=startup-script="$STARTUP" \  # train_sku110k.sh runs on first boot
#   --scopes=https://www.googleapis.com/auth/cloud-platform   # let the VM write to GCS + self-delete


### 2.1.4  IAM gotcha — bucket access for the VM

The VM uses the project's default Compute Engine service account. That SA needs **write access
to the results bucket** so the training script can `rsync` artifacts back. Granted **once per
project**:

In [ ]:
# One-time bucket binding so the default compute SA can write results back.
# Run from the laptop (not on the VM).
#
# !gcloud storage buckets add-iam-policy-binding gs://${PROJECT_ID}-sku110k-yolo \
#   --member=serviceAccount:$(gcloud projects describe ${PROJECT_ID} --format='value(projectNumber)')-compute@developer.gserviceaccount.com \
#   --role=roles/storage.objectAdmin
#
# (We learned the hard way — the first run trained successfully but the EXIT trap's
# rsync silently failed with a 403 and the VM kept running idle. Now baked into setup.)


## 2.2  Training script — `detection/train_sku110k.sh`

Runs **on the VM** as root, automatically, on first boot. It owns the entire lifecycle:
install → train → validate → sync → self-delete.

### 2.2.1  Setup + the cost-safety EXIT trap

The trap is the most important block: it guarantees the VM syncs its results to GCS and
deletes itself **on every exit path** — success, failure, or crash. Without it, a silent
crash leaves a $1/hr GPU running indefinitely.

In [ ]:
# !/bin/bash
# set -uo pipefail
#
# BUCKET="__BUCKET__"        # replaced by launch_vm.sh
# RUN_DIR=/opt/runs
# LOG="$RUN_DIR/train.log"
# mkdir -p "$RUN_DIR"
# : > "$LOG"                 # truncate log
#
# # ---- COST-SAFETY EXIT TRAP --------------------------------------------------
# # Fires no matter how the script exits — success, error, even SIGTERM.
# cleanup() {
#   rc=$?
#   echo "=== cleanup (exit $rc): syncing /opt/runs to GCS, then self-deleting ==="
#   gsutil -m rsync -r "$RUN_DIR" "$BUCKET/results/" || echo "WARN: gsutil sync failed"
#
#   # Read our own VM name + zone from the metadata server (no hard-coding).
#   NAME=$(curl -s -H "Metadata-Flavor: Google" \
#     http://metadata.google.internal/computeMetadata/v1/instance/name)
#   ZONE=$(curl -s -H "Metadata-Flavor: Google" \
#     http://metadata.google.internal/computeMetadata/v1/instance/zone | awk -F/ '{print $NF}')
#
#   echo "=== deleting self: $NAME in $ZONE ==="
#   gcloud compute instances delete "$NAME" --zone "$ZONE" --quiet
# }
# trap cleanup EXIT


### 2.2.2  Logging — write to a file, NOT through the metadata runner

Subtle but critical bug we hit and fixed: YOLO's progress bars use `\r` (no newline) and produce
multi-KB "lines". GCP's metadata script-runner pipes stdout through Go's `bufio.Scanner` with a
**64 KB token limit** → "token too long" → the runner kills the child process mid-training.

**Fix:** redirect all heavy output directly to a file (`>> "$LOG" 2>&1`) and only echo short
milestone messages to stdout. The scanner never sees the long lines.

In [ ]:
# # Wait for the NVIDIA driver to come up (DLVM preinstalls it; this is just a safety guard).
# until nvidia-smi >> "$LOG" 2>&1; do echo "waiting for GPU driver..."; sleep 15; done
# echo "GPU ready."
#
# # OpenCV (an Ultralytics dep) needs system OpenGL libs not in the base image.
# export DEBIAN_FRONTEND=noninteractive
# echo "installing system libs (libgl1, libglib2.0-0)..."
# apt-get update                               >> "$LOG" 2>&1 \
#   && apt-get install -y libgl1 libglib2.0-0  >> "$LOG" 2>&1
#
# echo "installing ultralytics..."
# pip install --upgrade "ultralytics>=8.2"     >> "$LOG" 2>&1
#
# # Less spammy progress bars — every 10s instead of every 100ms.
# export TQDM_MININTERVAL=10
# export PYTHONUNBUFFERED=1


### 2.2.3  Train — the core command

This is the **single command that produced our model**. Every flag below was deliberately
chosen for the L4 + SKU-110K combination.

In [ ]:
# echo "=== starting YOLOv8m training (output -> $LOG) ==="
#
# yolo detect train \
#   model=yolov8m.pt    \   # medium variant, COCO-pretrained (transfer learning)
#   data=SKU-110K.yaml  \   # auto-downloads + converts the dataset (Part 1)
#   epochs=50           \   # UPPER BOUND only — the time cap below usually stops first
#   time=9.5            \   # HARD 9.5 h budget: YOLO auto-scales epochs to fill it
#   imgsz=1280          \   # high resolution for tiny dense products (see 1.5)
#   batch=-1            \   # AutoBatch: pick largest batch fitting ~60% VRAM (chose 3)
#   cos_lr=True         \   # cosine learning-rate schedule
#   project=/opt/runs   \   # output dir
#   name=sku110k        \
#   exist_ok=True       \
#   >> "$LOG" 2>&1
#
# echo "=== training finished (exit $?) ==="


**Why `time=9.5` instead of a fixed epoch count?**
In Ultralytics, `time=` *overrides* `epochs` and auto-scales the number of epochs to consume
the entire budget — trading wall-clock for accuracy. On the L4 we got further into the schedule
(epoch 47 / 50) than we would have on a slower GPU in the same window. It also makes the run
**cost-deterministic**: ~9.5 h × ~$0.83/h ≈ **$8** per training run, no surprises.

### 2.2.4  Validate — produce the target metrics

In [ ]:
# echo "=== running final validation ==="
#
# yolo detect val \
#   model="$RUN_DIR/sku110k/weights/best.pt" \   # the best epoch's checkpoint
#   data=SKU-110K.yaml \
#   imgsz=1280 \                                  # same resolution as training
#   project="$RUN_DIR" \
#   name=sku110k_val \
#   exist_ok=True \
#   >> "$LOG" 2>&1


### 2.2.5  Extract the four target metrics into `metrics.json`

Parses the per-epoch CSV that Ultralytics writes and pulls out the final-epoch row. Wrapped in
`|| true` so a parse hiccup can never skip the cleanup trap.

In [ ]:
# python3 - "$RUN_DIR" >> "$LOG" 2>&1 <<'PY' || true
# import csv, json, os, sys
#
# rd = sys.argv[1]
# csvf = os.path.join(rd, "sku110k", "results.csv")
#
# # Some early rows have empty mAP50 cells (warmup); skip them.
# rows = [r for r in csv.DictReader(open(csvf))
#         if r.get("metrics/mAP50(B)", "").strip()]
# last = rows[-1]
#
# out = {
#     "mAP_50":    round(float(last["metrics/mAP50(B)"]), 4),
#     "mAP_50_95": round(float(last["metrics/mAP50-95(B)"]), 4),
#     "precision": round(float(last["metrics/precision(B)"]), 4),
#     "recall":    round(float(last["metrics/recall(B)"]), 4),
#     "epoch":     int(float(last["epoch"])),
# }
# json.dump(out, open(os.path.join(rd, "metrics.json"), "w"), indent=2)
# print(json.dumps(out, indent=2))
# PY
#
# echo "=== training + validation complete; metrics.json written ==="
# # Script ends here -> EXIT trap fires -> sync to GCS + delete VM.


### 2.2.6  What lands in GCS

After the EXIT trap fires:

```
gs://ehc-mgrandhi-bc801a-sku110k-yolo/results/
├── best.pt                     ← fine-tuned checkpoint  (49.7 MiB)  ← used in Part 3
├── last.pt                     ← last-epoch checkpoint  (49.7 MiB)
├── metrics.json                ← four target numbers
├── train.log                   ← full training log      (19.9 MiB)
├── sku110k/                    ← training run dir
│   ├── results.png             ← loss + mAP curves      ← used in Part 3
│   ├── results.csv             ← per-epoch metrics
│   ├── args.yaml               ← reproducibility record
│   ├── confusion_matrix.png
│   ├── BoxP_curve.png  BoxR_curve.png  BoxF1_curve.png  BoxPR_curve.png
│   ├── train_batch*.jpg        ← training-time augmentations preview
│   ├── val_batch*_pred.jpg     ← validation predictions
│   └── weights/{best,last}.pt
└── sku110k_val/                ← final val pass viz (same plots/images, eval-only)
```

---
# Part 3 — Before vs. After  ▶️ *(this part runs)*

Everything below executes. It pulls our fine-tuned `best.pt` from GCS and compares it against
the stock COCO model on the **same** shelf images — the visual proof of what training bought us.

## 3.1  Setup

In [ ]:
# Install Ultralytics (YOLOv8). Quiet to keep the log clean.
!pip install -q "ultralytics>=8.2"

import os, glob, json
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Confirm GPU if present (inference works fine on CPU too).
!nvidia-smi -L || echo 'No GPU - running inference on CPU (fine for a few images).'

## 3.2  Configuration

`PROJECT_ID` points the notebook at the GCS bucket the training job wrote to:
`gs://<PROJECT_ID>-sku110k-yolo/results/...`

In [ ]:
# >>> EDIT IF NEEDED <<<
PROJECT_ID = 'ehc-mgrandhi-bc801a'                  # same project used for training
BUCKET     = f'gs://{PROJECT_ID}-sku110k-yolo'

BEST_PT_GCS = f'{BUCKET}/results/best.pt'
METRICS_GCS = f'{BUCKET}/results/metrics.json'
CURVES_GCS  = f'{BUCKET}/results/sku110k/results.png'

CONF      = 0.25       # confidence threshold for display
IMGSZ     = 1280       # match training resolution (dense small objects)
N_SAMPLES = 6          # how many shelf images to compare

## 3.3  Load the two models

**Before** = stock COCO-pretrained YOLOv8m (auto-downloads).
**After**  = our fine-tuned `best.pt`, pulled from GCS (authenticate to Colab first). If you'd
rather not authenticate, use the `files.upload()` fallback cell to drag-and-drop `best.pt`.

In [ ]:
# BEFORE: COCO-pretrained YOLOv8m (auto-download).
before_model = YOLO('yolov8m.pt')
print('Loaded BEFORE model: yolov8m.pt (COCO-pretrained)')

In [ ]:
# AFTER: download our fine-tuned best.pt from GCS.
# Auth (Colab). Comment out if running outside Colab with gcloud already configured.
try:
    from google.colab import auth
    auth.authenticate_user()
    print('Authenticated to Google Cloud.')
except Exception as e:
    print('Not in Colab or auth skipped:', e)

!gsutil cp {BEST_PT_GCS} best.pt && echo 'Downloaded best.pt from GCS'

In [ ]:
# FALLBACK (only if the GCS download above failed): manually upload best.pt.
# from google.colab import files
# up = files.upload()        # choose best.pt
# print('Uploaded:', list(up.keys()))

In [ ]:
after_model = YOLO('best.pt')
print('Loaded AFTER model: best.pt (fine-tuned on SKU-110K)')

## 3.4  Sample shelf images

A few sample images to compare on. Swap in your own shelf photos via the `files.upload()` cell,
or copy SKU-110K test images into the bucket and pull them down.

In [ ]:
import urllib.request
os.makedirs('samples', exist_ok=True)

# A few public sample shelf images (replace with your own for a richer comparison).
SAMPLE_URLS = [
    'https://raw.githubusercontent.com/ultralytics/assets/main/im/grocery.jpg',
]
sample_paths = []
for i, url in enumerate(SAMPLE_URLS):
    p = f'samples/shelf_{i}.jpg'
    try:
        urllib.request.urlretrieve(url, p)
        sample_paths.append(p)
    except Exception as e:
        print('Could not fetch', url, e)

print('Fetched', len(sample_paths), 'sample(s).')
print('If you have none, run the upload cell below.')

In [ ]:
# Upload your own shelf images (recommended for a richer comparison).
# from google.colab import files
# up = files.upload()
# sample_paths = list(up.keys())

# Or, if you copied SKU-110K test images into the bucket:
# !gsutil -m cp 'gs://{}/samples/*.jpg'.format(PROJECT_ID + '-sku110k-yolo') samples/
# sample_paths = sorted(glob.glob('samples/*.jpg'))[:N_SAMPLES]

sample_paths = sample_paths[:N_SAMPLES]
assert sample_paths, 'No sample images! Use the upload cell above.'
sample_paths

## 3.5  Before vs. After — side by side

For each image we run both models and draw a 2-up panel: **left = COCO (before)**,
**right = fine-tuned (after)**, with the detection count in each title. This is the checkpoint's
punchline — the fine-tuned model finds the hundreds of densely-packed products the COCO model
misses entirely.

In [ ]:
def annotated(model, path):
    """Run inference and return (RGB annotated image, num_detections)."""
    r = model.predict(path, conf=CONF, imgsz=IMGSZ, verbose=False)[0]
    bgr = r.plot()                 # numpy array, BGR with boxes drawn
    rgb = bgr[:, :, ::-1]          # to RGB for matplotlib
    return rgb, len(r.boxes)

os.makedirs('panels', exist_ok=True)
panel_paths = []

for i, path in enumerate(sample_paths):
    before_img, n_before = annotated(before_model, path)
    after_img,  n_after  = annotated(after_model,  path)

    fig, axes = plt.subplots(1, 2, figsize=(18, 9))
    axes[0].imshow(before_img)
    axes[0].set_title(f'BEFORE - COCO yolov8m  .  {n_before} detections', fontsize=14)
    axes[0].axis('off')
    axes[1].imshow(after_img)
    axes[1].set_title(f'AFTER - fine-tuned on SKU-110K  .  {n_after} detections', fontsize=14)
    axes[1].axis('off')
    fig.suptitle(os.path.basename(path), fontsize=12)
    fig.tight_layout()

    out = f'panels/before_after_{i}.png'
    fig.savefig(out, dpi=120, bbox_inches='tight')
    panel_paths.append(out)
    plt.show()
    print(f'{os.path.basename(path)}:  before={n_before}  after={n_after}  '
          f'(+{n_after - n_before} more products found)')

## 3.6  Metrics summary (from the real GCP run)

The **honest** metrics from the full ~9.5 h L4 training, loaded from `metrics.json` in GCS — not
from this short demo.

In [ ]:
metrics = None
try:
    !gsutil cp {METRICS_GCS} metrics.json
    metrics = json.load(open('metrics.json'))
except Exception as e:
    print('Could not load metrics.json from GCS yet:', e)

if metrics:
    targets = {'mAP_50': 0.70, 'precision': 0.75, 'recall': 0.70}
    print(f"{'Metric':<14}{'Value':<10}{'Target':<10}{'Pass?'}")
    print('-' * 44)
    for k, label in [('mAP_50','mAP@0.5'), ('mAP_50_95','mAP@0.5:0.95'),
                     ('precision','Precision'), ('recall','Recall')]:
        v = metrics.get(k)
        t = targets.get(k)
        ok = '' if t is None else ('PASS' if v >= t else 'below')
        print(f"{label:<14}{v:<10}{(t if t else '-'):<10}{ok}")
    print(f"\nBest epoch: {metrics.get('epoch')}")
else:
    print('Run the GCP training job first, then re-run this cell.')

In [ ]:
# Training curves (loss / mAP over epochs) from the GCP run.
try:
    !gsutil cp {CURVES_GCS} results.png
    from IPython.display import Image, display
    display(Image('results.png'))
except Exception as e:
    print('Training curves not available yet:', e)

## 3.7  Save panels back to GCS (for the report)

Push the before/after panels into the bucket so they can be pulled into `report/figures/` later.

In [ ]:
for p in panel_paths:
    !gsutil cp {p} {BUCKET}/figures/{os.path.basename(p)}
print('Saved', len(panel_paths), 'panel(s) to', f'{BUCKET}/figures/')